In [ ]:
import pandas as pd
from pathlib import Path
import os

# 노트북 환경에서 프로젝트 루트 찾기
_PROJECT_ROOT = Path(os.getcwd())
while not (_PROJECT_ROOT / ".git").exists() and _PROJECT_ROOT != _PROJECT_ROOT.parent:
    _PROJECT_ROOT = _PROJECT_ROOT.parent

print(f"프로젝트 루트: {_PROJECT_ROOT}")

In [ ]:
# SIDO 코드 매핑
SIDO_MAP = {
    11: "서울특별시",
    26: "부산광역시",
    27: "대구광역시",
    28: "인천광역시",
    29: "광주광역시",
    30: "대전광역시",
    31: "울산광역시",
    36: "세종특별자치시",
    41: "경기도",
    42: "강원도",
    43: "충청북도",
    44: "충청남도",
    45: "전라북도",
    46: "전라남도",
    47: "경상북도",
    48: "경상남도",
    50: "제주특별자치도",
    51: "강원특별자치도",
    52: "전북특별자치도",
}

In [ ]:
# 1. 데이터 로드
print("데이터 로드 중...")

# 시간 정보가 있는 데이터
flood_area_df = pd.read_csv(_PROJECT_ROOT/"data/processed/flood_area.csv")
flooding_disaster_df = pd.read_csv(_PROJECT_ROOT/"data/processed/Flooding_Disaster.csv")
weather_rain_df = pd.read_csv(_PROJECT_ROOT/"data/processed/weather_rain_data.csv")

# 시간 정보가 없는 데이터
area_df = pd.read_csv(_PROJECT_ROOT/"data/processed/area.csv")
solid_pump_df = pd.read_csv(_PROJECT_ROOT/"data/processed/solid_pump_data.csv")
riv_pro_df = pd.read_csv(_PROJECT_ROOT/"data/processed/riv_pro_df.csv")

print("✓ 데이터 로드 완료")

In [ ]:
# 2. 시간 정보가 있는 데이터들 병합 (SIGUNGU + FLOOD_YEAR + FLOOD_MONTH 기준)
print("시간 정보가 있는 데이터 병합 중...")

merged_df = pd.merge(
    flood_area_df,
    flooding_disaster_df,
    on=["SIGUNGU", "FLOOD_YEAR", "FLOOD_MONTH"],
    how="outer"
)

merged_df = pd.merge(
    merged_df,
    weather_rain_df,
    on=["SIGUNGU", "FLOOD_YEAR", "FLOOD_MONTH"],
    how="outer"
)

print(f"✓ 시간 정보 병합 완료: {len(merged_df)}행")
print(f"  컬럼: {list(merged_df.columns)}")

In [ ]:
# 3. area.csv로 SIDO 정보 추가 (SIGUNGU 기준)
print("SIDO 정보 추가 중...")

merged_df = pd.merge(merged_df, area_df, on="SIGUNGU", how="left")

# SIDO 정보가 없는 경우 SIGUNGU 앞 2자리로 자동 매핑
missing_sido_mask = merged_df["SIDO"].isna()
merged_df.loc[missing_sido_mask, "SIDO"] = merged_df.loc[missing_sido_mask, "SIGUNGU"] // 1000
merged_df.loc[missing_sido_mask, "SIDO_NAME"] = merged_df.loc[missing_sido_mask, "SIDO"].map(SIDO_MAP)
merged_df.loc[missing_sido_mask, "SIGUNGU_NAME"] = "정보없음"

print(f"✓ SIDO 정보 추가 완료")
print(f"  SIDO 결측치: {merged_df['SIDO'].isna().sum()}개")

In [ ]:
merged_df.head()

In [ ]:
# 4. solid_pump_data 처리 및 병합 (SIGUNGU 기준)
print("배수펌프 데이터 처리 및 병합 중...")

# SIGUNGU별로 집계 (중복 제거 - DRAINAGE_GRD는 max, PUMP_CNT는 sum)
solid_pump_agg = solid_pump_df.groupby("SIGUNGU").agg({
    "DRAINAGE_GRD": "max",  # 배수등급은 최댓값
    "PUMP_CNT": "sum"        # 펌프 개수는 합계
}).reset_index()

print(f"  중복 제거: {len(solid_pump_df)} → {len(solid_pump_agg)}행")

merged_df = pd.merge(merged_df, solid_pump_agg, on="SIGUNGU", how="left")

print(f"✓ 배수펌프 데이터 병합 완료")

In [ ]:
# 5. riv_pro_df 처리 및 병합
print("하천 데이터 처리 및 병합 중...")

# 지역코드를 SIGUNGU로 변경
riv_pro_df = riv_pro_df.rename(columns={"지역코드": "SIGUNGU"})

# SIGUNGU별로 집계 (중복 제거 - 평균 사용)
riv_pro_df_agg = riv_pro_df.groupby("SIGUNGU").agg({
    "RIV_GRD": "mean",
    "RIV_DIS_MIN": "mean",
    "RIV_DIS_GRD": "mean"
}).reset_index()

merged_df = pd.merge(merged_df, riv_pro_df_agg, on="SIGUNGU", how="left")

print(f"✓ 하천 데이터 병합 완료")

In [ ]:
# 5-2 지역구별 면적 크기 피처 추가
from size import add_size_feature
merged_df = add_size_feature(merged_df)

In [ ]:
# 6. 필요한 컬럼만 선택하여 최종 데이터 생성
print("최종 데이터 생성 중...")

final_columns = [
    "SIDO", "SIGUNGU", "DAM_RAIN", "DEAD", "FLOOD_GRD", 
    "RIV_DIS_MIN", "RIV_GRD", "DRAINAGE_GRD", "PUMP_CNT", 
    "RAIN_TOTAL", "FLOOD_AREA", "FLOOD_YEAR", "FLOOD_MONTH", "AREA_SIZE"
]

# 컬럼 존재 여부 확인
available_columns = [col for col in final_columns if col in merged_df.columns]
missing_columns = [col for col in final_columns if col not in merged_df.columns]

if missing_columns:
    print(f"⚠ 누락된 컬럼: {missing_columns}")

final_df = merged_df[available_columns].copy()

# 데이터 타입 변환
int_columns = ['SIDO', 'SIGUNGU', 'DEAD', 'FLOOD_GRD', 'RIV_GRD', 'DRAINAGE_GRD', 'PUMP_CNT', 'FLOOD_YEAR', 'FLOOD_MONTH']
float_columns = ['DAM_RAIN', 'RIV_DIS_MIN', 'RAIN_TOTAL', 'FLOOD_AREA', 'AREA_SIZE']

for col in int_columns:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce').round().astype('Int64')

for col in float_columns:
    if col in final_df.columns:
        final_df[col] = pd.to_numeric(final_df[col], errors='coerce').astype('float64')

print(f"✓ 최종 데이터 생성 완료: {len(final_df)}행, {len(final_df.columns)}개 컬럼")

In [ ]:
# 7. 데이터 저장
output_path = _PROJECT_ROOT / "data/data.csv"
final_df.to_csv(output_path, index=False)

print(f"\n{'='*60}")
print(f"✓ 데이터 저장 완료!")
print(f"  파일 경로: {output_path}")
print(f"  총 행 수: {len(final_df):,}행")
print(f"  총 컬럼 수: {len(final_df.columns)}개")
print(f"{'='*60}")

In [ ]:
# 8. 결과 확인
print("\n데이터 미리보기:")
print(final_df.head(10))

print("\n\n데이터 정보:")
print(final_df.info())

print("\n\n기술 통계:")
print(final_df.describe())

print("\n\n결측치 확인:")
print(final_df.isnull().sum())

# 시간 정보가 없는 데이터 일관성 검증
print("\n\n시간 정보가 없는 데이터 일관성 검증:")
check = final_df.groupby("SIGUNGU").agg({
    "DRAINAGE_GRD": "nunique",
    "PUMP_CNT": "nunique", 
    "RIV_GRD": "nunique",
    "RIV_DIS_MIN": "nunique"
})
print(f"  DRAINAGE_GRD 최대 고유값: {check['DRAINAGE_GRD'].max()}")
print(f"  PUMP_CNT 최대 고유값: {check['PUMP_CNT'].max()}")
print(f"  RIV_GRD 최대 고유값: {check['RIV_GRD'].max()}")
print(f"  RIV_DIS_MIN 최대 고유값: {check['RIV_DIS_MIN'].max()}")
print("\n✓ 각 SIGUNGU별로 시간 정보가 없는 데이터가 일관되게 병합되었습니다!")